# 04 - Mesh tallies, plots and weight windows

**~30 minutes.**

### The story

The same run also produced two **mesh tallies**: the neutron and photon flux on a
10 x 10 x 10 grid covering the whole machine. Three things are expected of you now:

1. get the field into a form other people can use - ParaView, Excel, a CFD code;
2. produce the pictures that go into the report;
3. use the result to build **variance reduction** for the next, finer run.

### What you will use

`Meshtal`, `readMesh`, `Fmesh.print_info`, `.grid`, `write`, `write_all`, `collapse_grids`,
`transform_fmesh`, CuV meshes, `MeshPlotter`, `Atlas`, `WW`.

> **Solutions notebook.** Identical to `notebooks/04_mesh_tallies_plots_and_weight_windows.ipynb`, with the exercises filled in. Try them yourself first.

In [ ]:
# Run this cell first. It sets up the paths used by the whole notebook.
from pathlib import Path

DATA = Path("..") / "data"        # the MCNP files, read-only
OUT = Path("..") / "outputs"      # everything we produce goes here
OUT.mkdir(exist_ok=True)

%matplotlib inline
print("data folder :", DATA.resolve())
print("output folder:", OUT.resolve())

In [ ]:
import numpy as np
import pyvista as pv

from f4enix.output.meshtal.meshtal import Meshtal

## 1. Reading a meshtal

Building the `Meshtal` object only looks at the headers. `readMesh()` is what actually loads
the data - you can pass it a single ID or a list if the file is big and you only need one mesh.

In [ ]:
meshtal = Meshtal(DATA / "iter_1d" / "meshtal")
meshtal.readMesh()

print("mesh tallies in the file:", meshtal.mesh_ids)

In [ ]:
fmesh = meshtal.mesh[1004]

for key, value in fmesh.print_info().items():
    print(f"{key:10s}: {str(value)[:70]}")

## 2. The data itself

Two views of the same thing:

* `fmesh.data` - a raw numpy array, indexed `[time, energy, k, j, i, value/error]`;
* `fmesh.grid` - a **PyVista** grid, which is what you want for anything geometrical.

In [ ]:
print("raw array shape:", fmesh.data.shape)
print()
print(fmesh.grid)

In [ ]:
values = fmesh.grid["Value - Total"]
errors = fmesh.grid["Error - Total"]

print(f"voxels          : {len(values)}")
print(f"max flux        : {values.max():.3e}")
print(f"voxels scoring 0: {(values == 0).sum()}")
print(f"median rel error: {np.median(errors[values > 0]):.3f}")

> The mesh tallies here have a single energy and time bin. When they do not,
> `fmesh.get_etbin_data(ebin=..., tbin=...)` extracts one bin as a mesh of its own.

## 3. Exporting

`write` takes an **output folder** and a format. `vtk` is the one you want for ParaView
(a rectilinear mesh comes out as `.vtr`).

In [ ]:
fmesh.write(OUT, out_format="vtk")
fmesh.write(OUT, out_format="csv")

print([p.name for p in sorted(OUT.glob("Tally_1004*"))])

`point_cloud` and `ip_fluent` write the voxel centres as text - the formats you need to hand
results over to a thermal or CFD analyst.

`write_all` does the whole file in one call.

In [ ]:
vtk_dir = OUT / "meshtal_vtk"
vtk_dir.mkdir(exist_ok=True)

meshtal.write_all(vtk_dir)
print([p.name for p in sorted(vtk_dir.iterdir())])

## 4. Putting several mesh tallies on one grid

Two separate `.vtr` files means two separate loads in ParaView. If the mesh tallies share the
same grid, `collapse_grids` merges them into a single grid with one array per quantity - and
this is also where you give the arrays their real names and units.

In [ ]:
combined = meshtal.collapse_grids(
    {
        1004: ["neutron flux [n/cm3/s]", "neutron flux [rel err]"],
        1024: ["photon flux [p/cm3/s]", "photon flux [rel err]"],
    }
)

print(combined.array_names)
combined.save(OUT / "iter_1d_fields.vtr")

## 5. Mesh tallies that are not where they say they are

A `FMESH` card can carry a `tr=` flag, in which case the coordinates in the meshtal are in the
*transformed* frame. `transform_fmesh` applies the transformations from the corresponding input
file, so the mesh ends up where the geometry actually is.

In [ ]:
from f4enix.input.MCNPinput import Input

transformed = Meshtal(DATA / "extra" / "meshtal_transform")
transformed.readMesh()
tr_input = Input.from_input(DATA / "extra" / "transforms.i")

print("transformations available:", list(tr_input.transformations.keys()))
print("before:", transformed.mesh[2124].grid.bounds)

transformed.transform_fmesh(tr_input)

print("after :", transformed.mesh[2124].grid.bounds)

Note that the grid changed type as well: a rotated mesh is no longer rectilinear, so it becomes
a structured grid.

## 6. CuV meshes

D1SUNED can write **cell-under-voxel** meshes, where each voxel knows which cells contribute to
it. Reading one needs `filetype="CUV"` and a normalisation: `ctot` (per total volume) or
`celf` (per cell volume).

In [ ]:
cuv = Meshtal(DATA / "extra" / "meshtal_cuv", filetype="CUV")
cuv.readMesh(norm="ctot")

print("CuV mesh tallies:", cuv.mesh_ids)
print(cuv.mesh[44].grid)

With CuV there is a second source of uncertainty on top of the statistical one: the voxel
volumes are themselves sampled. F4Enix can compute that error and add it to the grid, so you
can look at both in ParaView.

In [ ]:
from f4enix.output.cuv_sampling_error import add_sampling_error_to_vtk

cuv_grid = add_sampling_error_to_vtk(
    cuv.mesh[44].grid,
    DATA / "extra" / "meshtal_cuv",
    voxel_sampling_points=1000,
)

print(cuv_grid.array_names)
sampling_error = cuv_grid["Volume Sampling Error"]
print(f"max volume sampling error: {sampling_error.max():.3e}")

## 7. Pictures

`MeshPlotter` slices a mesh and renders the slices. It can superimpose an STL of the geometry,
which is what makes the pictures readable.

**Units matter.** The mesh is in centimetres; this STL is in decimetres, so it has to be scaled
by 10 before the two can be overlaid. Always check the bounds.

In [ ]:
stl_raw = pv.read(DATA / "iter_1d" / "iter1D.stl")
stl = stl_raw.scale(10, inplace=False)

print("stl, as read :", stl_raw.bounds)
print("stl, scaled  :", stl.bounds)
print("mesh         :", fmesh.grid.bounds)

Three ways of slicing are available. `slice_on_axis` cuts the mesh at equally spaced positions
along one axis; `y` gives the poloidal cross-sections you normally show for a tokamak.

In [ ]:
from f4enix.output.plotter import Atlas, MeshPlotter

# a slice through an empty region should give an empty picture, not an error.
# this has to be set *after* importing the plotter module
pv.global_theme.allow_empty_mesh = True

plotter = MeshPlotter(fmesh.grid, stl=stl)

slices = plotter.slice_on_axis("y", 3)

for name, mesh_slice, stl_slice in slices:
    print(f"{name:20s} mesh cells: {mesh_slice.n_cells:5d}  stl cells: {stl_slice.n_cells}")

The other two are `slice_toroidal(30)` - vertical slices rotating around the z axis, the
natural choice for a full torus - and `slice(...)`, where you give an origin and a normal for
each slice.

`plot_slices` renders them off-screen and hands you back PIL images.

In [ ]:
images = plotter.plot_slices(
    slices,
    "Value - Total",
    scale_title="n/cm3/s",
    n_colors=8,
)

print([name for name, _ in images])
images[0][1]

## 8. The atlas

`Atlas` collects those images into a Word document, one section per quantity. This is the
deliverable most projects actually ask for.

In [ ]:
atlas = Atlas("neutron_atlas")
atlas.add_section("Neutron flux", images)
atlas.save(OUT)

print("written:", (OUT / "neutron_atlas.docx").is_file())

## 9. Weight windows

The last piece is variance reduction. `WW` reads and manipulates MCNP weight window files.

In [ ]:
from f4enix.input.ww_gvr import WW

ww = WW.load_from_ww_file(DATA / "ww" / "ww_simple_cart")
print(ww.info)

The important number in that summary is the **ratio**: how much a voxel's weight window differs
from its neighbours'. Large ratios are what produce the long histories that ruin a run - and
they are exactly what tally 74 was complaining about in notebook 03.

Four operations cover most of what you do to a weight window.

In [ ]:
ww.multiply(2.0)                        # global renormalisation
ww.soften(0.6)                          # flatten the distribution (values ** 0.6)
ww.mitigate_long_histories(max_ratio=5.0)   # zero the voxels with an extreme ratio

print(ww.info)

In [ ]:
# a neutron-only weight window can be extended to photons
print("before:", ww.particles)
ww.add_particle(norm=1.0, soft=1.0)
print("after :", ww.particles)

In [ ]:
ww.write_to_ww_file(OUT / "ww_modified")     # for MCNP
ww.export_as_vtk(OUT / "ww_modified.vts")    # for ParaView

print([p.name for p in sorted(OUT.glob("ww_modified*"))])

> `ww.plot()` opens an interactive 3-D view. It is commented out here because it blocks the
> notebook until you close the window - uncomment it if you want to explore.

In [ ]:
# ww.plot()

## 10. A GVR out of the mesh tally

The most useful thing in this module: turn the flux map you just looked at into a **global
variance reduction** mesh, ready for the next run. This is the MAGIC algorithm - the weight
window is built so that the weight is inversely proportional to the flux.

In [ ]:
gvr = WW.create_gvr_from_meshtally_file(
    DATA / "iter_1d" / "meshtal",
    tally_num=1004,
    maximum_splitting_ratio=5.0,
    softening_factor=1.0,
)

print(gvr.info)

Look at the maximum ratio: it is enormous, because 4% of the voxels scored nothing and end up
next to voxels that did. Mitigating brings it back under control.

In [ ]:
def max_ratio(weight_window):
    return max(
        np.max(array)
        for by_energy in weight_window.ratios.values()
        for array in by_energy.values()
    )


print(f"before mitigation: {max_ratio(gvr):.3e}")
gvr.mitigate_long_histories(max_ratio=10.0)
print(f"after mitigation : {max_ratio(gvr):.3e}")

In [ ]:
gvr.write_to_ww_file(OUT / "gvr_neutron")
gvr.export_as_vtk(OUT / "gvr_neutron.vts")

print([p.name for p in sorted(OUT.glob("gvr_neutron*"))])

---

# Exercises

## Exercise 1 - a photon GVR

Build the equivalent weight window for photons, out of mesh tally **1024**:

1. create the GVR with a maximum splitting ratio of 5;
2. soften it with a factor of **0.6**;
3. mitigate the long histories with a maximum ratio of **10**;
4. write `outputs/gvr_photon` (the MCNP file) and `outputs/gvr_photon.vts` (for ParaView).

**Hints**
* `WW.create_gvr_from_meshtally_file(path, tally_num=..., maximum_splitting_ratio=...)`;
* the three operations are methods on the object, applied in place;
* `write_to_ww_file` and `export_as_vtk`.

In [ ]:
# 1: same call as before, on the photon mesh tally
photon_gvr = WW.create_gvr_from_meshtally_file(
    DATA / "iter_1d" / "meshtal",
    tally_num=1024,
    maximum_splitting_ratio=5.0,
)

# 2: flatten the distribution a bit
photon_gvr.soften(0.6)

# 3: kill the voxels that would cause long histories
photon_gvr.mitigate_long_histories(max_ratio=10.0)

# 4: one file for MCNP, one for ParaView
photon_gvr.write_to_ww_file(OUT / "gvr_photon")
photon_gvr.export_as_vtk(OUT / "gvr_photon.vts")

print(photon_gvr.info)

In [ ]:
# --- self-check, do not modify ---
assert (OUT / "gvr_photon").is_file(), "the weight window file was not written"
assert (OUT / "gvr_photon.vts").is_file(), "the VTK file was not written"
assert max_ratio(photon_gvr) <= 10.0 + 1e-6, (
    f"the maximum ratio is still {max_ratio(photon_gvr):.3e} - did you mitigate?"
)
print("Exercise 1 correct")

## Exercise 2 - a photon atlas

Produce the report figure for the photon flux:

1. take mesh tally 1024 out of `meshtal`;
2. build a `MeshPlotter` for it, with the same scaled `stl`;
3. cut it into **three slices along z**;
4. render the slices with `plot_slices`, using the array `"Value - Total"`;
5. put them in an `Atlas` called `photon_atlas` and save it into `OUT`.

**Hints**
* `meshtal.mesh[1024]` gives you the `Fmesh`, and `.grid` its PyVista grid;
* `MeshPlotter(grid, stl=stl)`, then `slice_on_axis("z", 3)`;
* `Atlas("photon_atlas")`, then `add_section("Photon flux", images)`, then `save(OUT)`.

In [ ]:
# 1: the photon mesh tally
photon_fmesh = meshtal.mesh[1024]

# 2: same STL, new mesh
photon_plotter = MeshPlotter(photon_fmesh.grid, stl=stl)

# 3: three cuts along the vertical axis
photon_slices = photon_plotter.slice_on_axis("z", 3)

# 4: render
photon_images = photon_plotter.plot_slices(
    photon_slices,
    "Value - Total",
    scale_title="p/cm3/s",
)

# 5: assemble the report
photon_atlas = Atlas("photon_atlas")
photon_atlas.add_section("Photon flux", photon_images)
photon_atlas.save(OUT)

print("slices:", len(photon_slices))

In [ ]:
# --- self-check, do not modify ---
assert (OUT / "photon_atlas.docx").is_file(), "photon_atlas.docx was not written"
assert len(photon_slices) == 3, f"expected 3 slices, got {len(photon_slices)}"
assert len(photon_images) == 3, f"expected 3 images, got {len(photon_images)}"
print("Exercise 2 correct")

---

## What you can now do

| Question | Method |
|----------|--------|
| what is in this meshtal? | `Meshtal`, `readMesh`, `Fmesh.print_info` |
| give me the numbers | `fmesh.data`, `fmesh.grid["Value - Total"]` |
| I need it in ParaView / Excel / Fluent | `fmesh.write`, `meshtal.write_all` |
| one file with everything | `collapse_grids` |
| the mesh is in the wrong place | `transform_fmesh` |
| cell-under-voxel results | `filetype="CUV"`, `add_sampling_error_to_vtk` |
| I need pictures | `MeshPlotter.slice_on_axis` + `plot_slices` |
| I need a Word report | `Atlas` |
| my run has long histories | `WW.mitigate_long_histories` |
| I need variance reduction for the next run | `WW.create_gvr_from_meshtally_file` |

---

## That is the whole workflow

```
01  read and edit the model
02  materials, libraries, tallies, run size
                |
                v   (MCNP runs)
03  is the run trustworthy? what did it score?
04  map the field, report it, and feed the next run
```

Everything you did here is scriptable, which is the real point: the same twenty lines run on
one model or on fifty.

Full documentation: <https://f4enix.readthedocs.io/en/latest/>